In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

# =====================================================
# 1. IMPORT LIBRARIES
# =====================================================

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

plt.style.use('dark_background')
import plotly.io as pio

pio.renderers.default = 'iframe'

In [ ]:
# =====================================================
# 2. LOAD DATA
# =====================================================
file_path = "/kaggle/input/datasets/krupalpatel07/bank-of-america-daily-historical-data/bofa.csv"
df = pd.read_csv(file_path)


In [ ]:
# =====================================================
# 3. PREPROCESSING
# =====================================================
df.columns = [c.lower() for c in df.columns]
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.set_index('date', inplace=True)


In [ ]:
# =====================================================
# 4. HTML CYBER HEADER
# =====================================================
from IPython.display import display, HTML

def cyber_header(text):
    display(HTML(f"""
    <div style="
        background: repeating-linear-gradient(45deg, #061a2b, #061a2b 10px, #082a44 10px, #082a44 20px);
        border: 1px solid #00e5ff;
        box-shadow: 0 0 12px #00e5ff inset, 0 0 12px #00e5ff;
        padding: 18px; border-radius: 14px; margin-top: 18px;">
        <h1 style="color:#7df9ff; text-align:center; letter-spacing:1px;">{text}</h1>
    </div>
    """))

cyber_header("📊 Price + Liquidity Blueprint")

In [ ]:
# =====================================================
# 5. CANDLE + VOLUME PROFILE (PROXY)
# =====================================================
fig = go.Figure()
fig.add_trace(go.Candlestick(x=df.index, open=df['open'], high=df['high'], low=df['low'], close=df['close']))
fig.add_trace(go.Bar(x=df.index, y=df['volume'], name='Volume', opacity=0.3))
fig.update_layout(template='plotly_dark', title='Price & Volume Overlay')
fig.show()

In [ ]:
# =====================================================
# 7. REALIZED VOLATILITY & VOL CLUSTERS
# =====================================================
cyber_header("🌪️ Realized Volatility Clusters")

df['ret'] = df['close'].pct_change()
df['rv_10'] = df['ret'].rolling(10).std() * np.sqrt(252)

fig = px.line(df, y='rv_10', title='10D Realized Vol (Annualized)')
fig.show()

In [ ]:
# =====================================================
# 8. IMPLIED VOL PROXY (RANGE-BASED)
# =====================================================
cyber_header("🧪 Implied Vol Proxy (Parkinson)")

# Parkinson volatility proxy
df['parkinson'] = np.sqrt((1/(4*np.log(2))) * (np.log(df['high']/df['low'])**2)).rolling(10).mean() * np.sqrt(252)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['rv_10'], name='Realized Vol'))
fig.add_trace(go.Scatter(x=df.index, y=df['parkinson'], name='Range Vol Proxy'))
fig.update_layout(template='plotly_dark')
fig.show()

In [ ]:
# =====================================================
# 9. REGIME SWITCHING (GMM AS HMM-LITE)
# =====================================================
cyber_header("🧠 Regime Switching (GMM)")

feat = df[['ret','rv_10']].dropna()
scaler = StandardScaler()
X = scaler.fit_transform(feat)

# 3 regimes: low/med/high vol
model = GaussianMixture(n_components=3, random_state=42)
labels = model.fit_predict(X)

reg = pd.Series(labels, index=feat.index, name='regime')

fig = px.scatter(x=feat.index, y=feat['rv_10'], color=reg.astype(str), title='Volatility Regimes')
fig.show()


In [ ]:
# =====================================================
# 10. LIQUIDITY GAPS (FAIR VALUE GAP PROXY)
# =====================================================
cyber_header("🧩 Liquidity Gaps (FVG Proxy)")

# Simple FVG proxy: gap between prev high and next low
df['prev_high'] = df['high'].shift(1)
df['gap_down'] = df['prev_high'] < df['low']

fig = px.scatter(df, x=df.index, y='close', color='gap_down', title='Gap Detection')
fig.show()


In [ ]:
# =====================================================
# 11. STRATEGY SKETCH (VOL + FLOW)
# =====================================================
cyber_header("🎯 Strategy Sketch")

# Idea: trade when flow aligns with low-vol expansion
signal = (df['oflow'] > 0) & (df['rv_10'] < df['rv_10'].rolling(50).mean())
df['signal'] = signal.astype(int)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['close'], name='Close'))
fig.add_trace(go.Scatter(x=df.index[df['signal']==1], y=df['close'][df['signal']==1],
                         mode='markers', name='Long Signal'))
fig.update_layout(template='plotly_dark')
fig.show()

In [ ]:
# =====================================================
# 12. FINAL NOTES
# =====================================================
cyber_header("📌 Takeaways")

print("""
1. Orderflow proxy highlights aggressor dominance days.
2. Realized vs range volatility divergence can signal expansion.
3. GMM regimes segment market into volatility states.
4. Liquidity gaps often align with continuation.
5. Combining flow + low-vol compression offers entry edges.
""")

# =====================================================
# END
# =====================================================